In [1]:
from cmdstanpy import CmdStanModel
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats as st
import os

from joblib import Parallel, delayed
from tqdm.auto import tqdm

#### Compile Stan model

In [3]:
stan_path = Path("/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/stan/NB_regression_puradj.stan")
model = CmdStanModel(stan_file=str(stan_path))

#### Fit one gene

In [5]:
def fit_one_gene(
    gene_df: pd.DataFrame,
    model: CmdStanModel,
    gene: str | None = None,      # optional convenience
    cna: str = "all",             # "amp" | "del" | "all"
    et: float = 0.15,
    min_aneup: int = 5,
    min_unique_counts: int = 5,
    min_cn_abs_sum: float = 1.0,  # identifiability filter for cna="all"
    chains: int = 4,
    iter_warmup: int = 1000,
    iter_sampling: int = 1000,
    seed: int = 1,
    show_progress: bool = False,
    adapt_delta: float = 0.99,
    max_treedepth: int = 15,
):
    """
    Fit your Stan NB regression model for a single gene.
    Expects gene_df to be filtered to one gene OR pass gene and full df.

    Required columns in gene_df:
      gene, expr, copies, purity, stroma, sf, eup_dev_cancer, eup_equiv_cancer
    Optional columns:
      covar (if missing -> set to 'ALL')
    """
    # ---- subset to one gene if gene provided and gene_df contains multiple genes
    df = gene_df.copy()
    if gene is not None and "gene" in df.columns and df["gene"].nunique() > 1:
        df = df.loc[df["gene"] == gene].copy()
    if gene is None and "gene" in df.columns and df["gene"].nunique() == 1:
        gene = str(df["gene"].iloc[0])

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_rows_for_gene"}

    required = {"expr","copies","purity","stroma","sf","eup_dev_cancer","eup_equiv_cancer"}
    missing = required - set(df.columns)
    if missing:
        return {"status": "error", "gene": gene, "reason": f"missing_columns: {sorted(missing)}"}

    # ---- CNA subset (optional)
    if cna == "amp":
        df = df[df["copies"] > (2 - et)]
    elif cna == "del":
        df = df[df["copies"] < (2 + et)]
    elif cna == "all":
        pass
    else:
        raise ValueError("cna must be 'amp', 'del', or 'all'")

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_samples_after_cna_filter"}

    # ---- basic data QC
    df = df.dropna(subset=["expr","sf","purity","stroma","eup_dev_cancer","eup_equiv_cancer"])
    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "all_na_after_dropna"}

    if (df["expr"] < 0).any():
        return {"status": "error", "gene": gene, "reason": "negative_counts"}

    if not df["purity"].between(0, 1).all():
        return {"status": "error", "gene": gene, "reason": "purity_out_of_bounds"}

    if not (df["sf"] > 0).all():
        return {"status": "error", "gene": gene, "reason": "nonpositive_sf"}

    if not (df["eup_equiv_cancer"] > 0).all():
        return {"status": "error", "gene": gene, "reason": "nonpositive_eup_equiv_cancer"}

    # ---- skip degenerate genes early (prevents phi->inf + treedepth explosions)
    if df["expr"].nunique() < min_unique_counts:
        return {"status": "skipped", "gene": gene, "reason": "too_few_unique_counts"}

    # aneuploid count check (use copies as cancer_copies analogue)
    n_aneup = int((np.abs(df["copies"].astype(float) - 2.0) > (1.0 - et)).sum())
    if n_aneup < min_aneup or (df["expr"] == 0).all():
        return {"status": "skipped", "gene": gene, "n_aneup": n_aneup, "reason": "low_aneup_or_all_zero"}

    # identifiability check for cna="all": need some CN deviation mass
    if cna == "all" and df["eup_dev_cancer"].abs().sum() < min_cn_abs_sum:
        return {"status": "skipped", "gene": gene, "n_aneup": n_aneup, "reason": "too_little_cn_variation"}

    # ---- within one cohort: enforce K=1, covar='ALL'
    df["covar"] = "ALL"
    covar_levels = ["ALL"]
    covar_idx = np.ones(len(df), dtype=int)

    # ---- per-gene sf scaling (matches R script)
    mean_expr = float(df["expr"].mean())
    df["sf_scaled"] = df["sf"].astype(float) * mean_expr

    stan_data = {
        "N": int(len(df)),
        "y": df["expr"].astype(int).to_numpy(),
        "K": 1,
        "covar": covar_idx,
        "sf": df["sf_scaled"].to_numpy(dtype=float),
        "purity": df["purity"].to_numpy(dtype=float),
        "stroma": df["stroma"].to_numpy(dtype=float),
        "eup_equiv_cancer": df["eup_equiv_cancer"].to_numpy(dtype=float),
        "eup_dev_cancer": df["eup_dev_cancer"].to_numpy(dtype=float),
    }

    rng = np.random.default_rng(seed)
    init = {
        "b_scaling": rng.uniform(0.5, 1.5, size=1).tolist(),
        "b_noncancer": rng.uniform(0.5, 1.5, size=1).tolist(),
        "b_deviation": 0.0,
        "phi": 1.0,
    }

    fit = model.sample(
        data=stan_data,
        chains=chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        inits=init,
        show_progress=show_progress,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
    )

    draws = fit.draws_pd()

    # ---- extract posterior summaries
    # b_scaling[1] column name can be "b_scaling[1]" (CmdStan) or similar
    scaling_col = next((c for c in draws.columns if c.startswith("b_scaling")), None)
    if scaling_col is None:
        return {"status": "error", "gene": gene, "reason": "missing_b_scaling_draws"}

    if "b_deviation" not in draws.columns:
        return {"status": "error", "gene": gene, "reason": "missing_b_deviation_draws"}

    phi_col = "phi" if "phi" in draws.columns else None

    b_scaling = draws[scaling_col].to_numpy()
    b_dev = draws["b_deviation"].to_numpy()

    # z and p (same as you did)
    z_comp = float(b_dev.mean() / b_dev.std(ddof=1))
    p_value = float(2.0 * (1.0 - st.norm.cdf(abs(z_comp))))

    summ = fit.summary()

    # robust extraction of Rhat / ESS
    rhat_col = next((c for c in ["R_hat", "Rhat"] if c in summ.columns), None)
    ess_col  = next((c for c in ["Ess_bulk", "ESS_bulk", "N_Eff", "Ess"] if c in summ.columns), None)

    rhat_dev = float(summ.loc["b_deviation", rhat_col]) if (rhat_col and "b_deviation" in summ.index) else np.nan
    ess_dev  = float(summ.loc["b_deviation", ess_col]) if (ess_col and "b_deviation" in summ.index) else np.nan

    # mark borderline fits as warn (you can filter later)
    status = "ok"
    if (not np.isnan(rhat_dev) and rhat_dev > 1.05) or (not np.isnan(ess_dev) and ess_dev < 200):
        status = "warn"

    out = {
        "status": status,
        "gene": gene,
        "N": int(len(df)),
        "n_aneup": n_aneup,
        "cna": cna,
        # posterior summaries
        "mean_b_scaling": float(b_scaling.mean()),
        "sd_b_scaling": float(b_scaling.std(ddof=1)),
        "mean_b_deviation": float(b_dev.mean()),
        "sd_b_deviation": float(b_dev.std(ddof=1)),
        "z_comp": z_comp,
        "p_value": p_value,
        # optional
        "mean_phi": float(draws[phi_col].mean()) if phi_col else np.nan,
        "Rhat_b_deviation": rhat_dev,
        "ess_b_deviation": ess_dev,
        "covar_levels": covar_levels,
    }
    return out

In [7]:
DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/data/stan_model_test/"
df_long = pd.read_csv(os.path.join(DATA_PATH, "crc_joint_long.csv"))
df_long.head()

,gene,sample,expr,copies,CNstate,stroma,purity,sf,eup_dev_cancer,eup_equiv_cancer
0,AAR2,CRC-SW-U0085-T,1075,3,Loss,0.17,0.83,0.813161,0.5,1.5
1,AAR2,CRC-SW-U0125-T,1474,3,Gain,0.40,0.60,1.011400,0.5,1.5
2,AAR2,CRC-SW-U0225-T,1216,3,Loss,0.21,0.79,0.966235,0.5,1.5
3,AAR2,CRC-SW-U0290-T,1044,2,Loss,0.42,0.58,1.108472,0.0,1.0
4,AAR2,CRC-SW-U0308-T,1180,2,Gain,0.44,0.56,1.058325,0.0,1.0


In [11]:
# Or prefilter once and pass the slice
gene_df = df_long[df_long["gene"] == "ABHD3"]
res_singleGene = fit_one_gene(gene_df, 
                   model, 
                   et=0.15, 
                   min_aneup=5,
                   cna="all")

20:11:24 - cmdstanpy - INFO - CmdStan start processing
20:11:24 - cmdstanpy - INFO - Chain [1] start processing
20:11:24 - cmdstanpy - INFO - Chain [2] start processing
20:11:24 - cmdstanpy - INFO - Chain [3] start processing
20:11:24 - cmdstanpy - INFO - Chain [4] start processing
20:11:25 - cmdstanpy - INFO - Chain [4] done processing
20:11:25 - cmdstanpy - INFO - Chain [1] done processing
20:19:48 - cmdstanpy - INFO - Chain [2] done processing
20:21:08 - cmdstanpy - INFO - Chain [3] done processing
20:21:08 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 2 had 999 iterations at max treedepth (99.9%)
	Chain 3 had 999 iterations at max treedepth (99.9%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [13]:
res_singleGene

{'status': 'warn',
 'gene': 'ABHD3',
 'N': 53,
 'n_aneup': 20,
 'cna': 'all',
 'mean_b_scaling': 1.00312017025,
 'sd_b_scaling': 0.2113520601139657,
 'mean_b_deviation': 0.0365308886885,
 'sd_b_deviation': 0.1199044461155076,
 'z_comp': 0.30466667310492124,
 'p_value': 0.7606200345849357,
 'mean_phi': 7.7161172625,
 'Rhat_b_deviation': 1.04694,
 'ess_b_deviation': 37.9134,
 'covar_levels': ['ALL']}

In [11]:
# Fit a set of genes (do a small subset first)
genes = df_long["gene"].unique()[:10]

results = []
for g in genes:
    gene_df = df_long[df_long["gene"] == g]
    results.append(
        fit_one_gene(
            gene_df, model,
            cna="all",
            et=0.15,
            min_aneup=5,
            chains=4,
            iter_warmup=1000,
            iter_sampling=1000,
            adapt_delta=0.99,
            max_treedepth=15,
        )
    )

res_ = pd.DataFrame(results)

14:13:30 - cmdstanpy - INFO - CmdStan start processing
14:13:30 - cmdstanpy - INFO - Chain [1] start processing
14:13:30 - cmdstanpy - INFO - Chain [2] start processing
14:13:30 - cmdstanpy - INFO - Chain [3] start processing
14:13:30 - cmdstanpy - INFO - Chain [4] start processing
14:14:10 - cmdstanpy - INFO - Chain [3] done processing
14:14:13 - cmdstanpy - INFO - Chain [4] done processing
14:15:33 - cmdstanpy - INFO - Chain [1] done processing
14:16:00 - cmdstanpy - INFO - Chain [2] done processing
14:16:00 - cmdstanpy - INFO - CmdStan start processing
14:16:00 - cmdstanpy - INFO - Chain [1] start processing
14:16:00 - cmdstanpy - INFO - Chain [2] start processing
14:16:00 - cmdstanpy - INFO - Chain [3] start processing
14:16:00 - cmdstanpy - INFO - Chain [4] start processing
14:16:00 - cmdstanpy - INFO - Chain [2] done processing
14:16:00 - cmdstanpy - INFO - Chain [3] done processing
14:16:00 - cmdstanpy - INFO - Chain [4] done processing
14:16:00 - cmdstanpy - INFO - Chain [1] do

In [13]:
res_

,status,gene,N,n_aneup,cna,mean_b_scaling,sd_b_scaling,mean_b_deviation,sd_b_deviation,z_comp,p_value,mean_phi,Rhat_b_deviation,ess_b_deviation,covar_levels
0,ok,AAR2,53,39,all,0.729129,0.110758,0.170707,0.159056,1.073246,0.283161,14.470120,1.00100,1589.11778,[ALL]
1,ok,ABCB5,53,16,all,0.844775,0.478981,-0.022533,0.199768,-0.112794,0.910194,0.303112,0.99943,2809.60779,[ALL]
2,ok,ABCC4,53,35,all,0.638295,0.144331,-0.118917,0.190807,-0.623229,0.533134,5.361753,1.01386,642.29223,[ALL]
3,warn,ABCD1,53,15,all,0.737661,0.151943,0.132870,0.132252,1.004668,0.315057,10.538742,1.15545,33.32328,[ALL]
4,warn,ABHD3,53,20,all,1.003120,0.211352,0.036531,0.119904,0.304667,0.760620,7.716117,1.04694,37.91340,[ALL]
5,warn,ABHD12,53,24,all,0.972028,0.103063,0.276360,0.127985,2.159323,0.030825,11.608608,1.06947,71.46261,[ALL]
6,ok,ABHD13,53,34,all,0.712081,0.104075,0.272209,0.145941,1.865196,0.062154,11.274407,1.03704,201.48390,[ALL]
7,warn,ABR,53,20,all,1.615266,0.159778,-0.221596,0.188887,-1.173166,0.240729,11.028031,1.88269,3.26313,[ALL]
8,warn,ACAA2,53,26,all,1.419203,0.107955,-0.182169,0.177632,-1.025541,0.305108,10.162418,6.99935,2.09672,[ALL]
9,warn,ACADVL,53,22,all,1.403541,0.233381,-0.251208,0.103969,-2.416173,0.015685,10.917653,1.29770,5.31888,[ALL]


In [19]:
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    pv = pvals[ok]
    n = pv.size
    if n == 0:
        return out
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    tmp = np.empty_like(adj)
    tmp[order] = adj
    out[ok] = tmp
    return out

In [37]:
def postprocess_nb_results(
    res_df: pd.DataFrame,
    alpha: float = 0.05,
    comp_thr: float = 0.3,
    min_cn_signal: float = 0.05,   # minimum CN signal proxy 
    min_scaling: float = 1e-6,     # avoid division issues
):
    """
    Post-process Stan NB-regression results with columns:
      status, gene, N, n_aneup, cna, mean_b_scaling, sd_b_scaling,
      mean_b_deviation, sd_b_deviation, z_comp, p_value, mean_phi,
      Rhat_b_deviation, ess_b_deviation, covar_levels

    Adds:
      - adj_p : BH FDR over usable genes (status in {ok,warn})
      - comp_score, signed_comp
      - label_nb in {DCG, DSG, HYPER, OTHER, SKIP, ERROR}
    """

    df = res_df.copy()

    # Expected columns check 
    expected = {
        "status","gene","N","n_aneup","cna","mean_b_scaling","mean_b_deviation","p_value"
    }
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {sorted(missing)}")

    # Define usable genes: include ok + warn 
    usable = df["status"].isin(["ok", "warn"]) & df["p_value"].notna()

    # Default labels
    df["label_nb"] = "OTHER"
    df.loc[df["status"].str.lower().isin(["skip", "skipped"]), "label_nb"] = "SKIP"
    df.loc[df["status"].str.lower().isin(["error", "failed"]), "label_nb"] = "ERROR"

    # FDR over usable genes 
    df["adj_p"] = np.nan
    if usable.any():
        df.loc[usable, "adj_p"] = bh_fdr(df.loc[usable, "p_value"].to_numpy(dtype=float))

    # Scores
    df["comp_score"] = np.nan
    df["signed_comp"] = np.nan

    ok2 = (
        usable
        & df["mean_b_scaling"].notna()
        & df["mean_b_deviation"].notna()
        & (df["mean_b_scaling"].abs() > min_scaling)
    )

    df.loc[ok2, "comp_score"] = df.loc[ok2, "mean_b_deviation"] / df.loc[ok2, "mean_b_scaling"]

    # Sign handling:
    # If you ran fits separately for amp/del, you can flip sign for deletions if desired.
    # For cna="all" we don't have direction in this results table, so keep as comp_score.
    
    sign = np.ones(df.loc[ok2].shape[0], dtype=float)
    if "cna" in df.columns:
        # Optional convention: del -> -1, amp -> +1, all -> +1 (no direction info)
        cna_vals = df.loc[ok2, "cna"].astype(str).str.lower().to_numpy()
        sign[cna_vals == "del"] = -1.0
        # amp/all remain +1
    df.loc[ok2, "signed_comp"] = df.loc[ok2, "comp_score"].to_numpy(dtype=float) * sign

    # CN-signal gate proxy using fraction aneuploid 
    # Without per-gene CN summaries in res_df, use n_aneup/N as a proxy for CN informativeness.
    df["aneup_frac"] = np.nan
    df.loc[usable, "aneup_frac"] = (df.loc[usable, "n_aneup"] / df.loc[usable, "N"]).astype(float)

    cn_ok = ok2 & (df["aneup_frac"] >= min_cn_signal)

    # Calls 
    is_sig = cn_ok & (df["adj_p"] <= alpha)
    is_nonsig = cn_ok & (df["adj_p"] > alpha)

    # DCG: significant negative deviation (compensation)
    df.loc[is_sig & (df["signed_comp"] <= -comp_thr), "label_nb"] = "DCG"

    # Hyperactivation: significant positive deviation (optional)
    df.loc[is_sig & (df["signed_comp"] >= comp_thr), "label_nb"] = "DCG"

    # DSG: no significant deviation + close to CN expectation
    df.loc[is_nonsig & (df["signed_comp"].abs() <= comp_thr), "label_nb"] = "DSG"

    # Optional: flag WARN calls as low-confidence (uncomment if useful)
    # df.loc[df["status"] == "warn", "label_nb"] = df.loc[df["status"] == "warn", "label_nb"] + "_LOWCONF"

    return df

In [39]:
# Post-process into DSG/DCG calls
res_pp = postprocess_nb_results(res_, 
                                alpha=0.05, 
                                comp_thr=0.3, 
                                min_cn_signal=0.05)
res_pp

,status,gene,N,n_aneup,cna,mean_b_scaling,sd_b_scaling,mean_b_deviation,sd_b_deviation,z_comp,p_value,mean_phi,Rhat_b_deviation,ess_b_deviation,covar_levels,label_nb,adj_p,comp_score,signed_comp,aneup_frac
0,ok,AAR2,53,39,all,0.729129,0.110758,0.170707,0.159056,1.073246,0.283161,14.470120,1.00100,1589.11778,[ALL],DSG,0.450081,0.234124,0.234124,0.735849
1,ok,ABCB5,53,16,all,0.844775,0.478981,-0.022533,0.199768,-0.112794,0.910194,0.303112,0.99943,2809.60779,[ALL],DSG,0.910194,-0.026673,-0.026673,0.301887
2,ok,ABCC4,53,35,all,0.638295,0.144331,-0.118917,0.190807,-0.623229,0.533134,5.361753,1.01386,642.29223,[ALL],DSG,0.666418,-0.186303,-0.186303,0.660377
3,warn,ABCD1,53,15,all,0.737661,0.151943,0.132870,0.132252,1.004668,0.315057,10.538742,1.15545,33.32328,[ALL],DSG,0.450081,0.180123,0.180123,0.283019
4,warn,ABHD3,53,20,all,1.003120,0.211352,0.036531,0.119904,0.304667,0.760620,7.716117,1.04694,37.91340,[ALL],DSG,0.845133,0.036417,0.036417,0.377358
5,warn,ABHD12,53,24,all,0.972028,0.103063,0.276360,0.127985,2.159323,0.030825,11.608608,1.06947,71.46261,[ALL],DSG,0.154126,0.284313,0.284313,0.452830
6,ok,ABHD13,53,34,all,0.712081,0.104075,0.272209,0.145941,1.865196,0.062154,11.274407,1.03704,201.48390,[ALL],OTHER,0.207180,0.382272,0.382272,0.641509
7,warn,ABR,53,20,all,1.615266,0.159778,-0.221596,0.188887,-1.173166,0.240729,11.028031,1.88269,3.26313,[ALL],DSG,0.450081,-0.137189,-0.137189,0.377358
8,warn,ACAA2,53,26,all,1.419203,0.107955,-0.182169,0.177632,-1.025541,0.305108,10.162418,6.99935,2.09672,[ALL],DSG,0.450081,-0.128360,-0.128360,0.490566
9,warn,ACADVL,53,22,all,1.403541,0.233381,-0.251208,0.103969,-2.416173,0.015685,10.917653,1.29770,5.31888,[ALL],DSG,0.154126,-0.178982,-0.178982,0.415094
